# Training Loops
scope:
- typical training loop
- training with gradient accumulation
- mixed precision training
- distributed training
- model parallelism
- pipeline parallelism
- data parallelism
- tensor parallelism
  


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


## 1. Typical Training Loop

In [3]:
# Epoch-based traing loop with learning rate scheduler
# --- 1. Prepare dummy dataset ---
X = torch.randn(100, 10)  # 100 samples, 10 features each
y = torch.randint(0, 2, (100,))  # Binary classification labels
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# --- 2. Define model ---
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 2)  # Output 2 classes

    def forward(self, x):
        return self.fc(x)

model = SimpleNet()

# --- 3. Define loss and optimizer ---
num_epochs = 5

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# --- 4. Training loop ---
for epoch in range(num_epochs):
    running_loss = 0.0

    for batch_X, batch_y in dataloader:
        # 1) Zero gradients from previous step
        optimizer.zero_grad()

        # 2) Forward pass
        outputs = model(batch_X)

        # 3) Compute loss
        loss = criterion(outputs, batch_y)

        # 4) Backward pass
        loss.backward()

        # 5) Update parameters
        optimizer.step()

        running_loss += loss.item()

    # Update learning rate per epoch
    lr_scheduler.step()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(dataloader):.4f}")

print("Training complete.")

Epoch [1/5], Loss: 0.6906
Epoch [2/5], Loss: 0.7091
Epoch [3/5], Loss: 0.7100
Epoch [4/5], Loss: 0.6725
Epoch [5/5], Loss: 0.7043
Training complete.


In [11]:
# step-based training loop with learning rate scheduler
# --- 1. Prepare dummy dataset ---
X = torch.randn(100, 10)  # 100 samples, 10 features each
y = torch.randint(0, 2, (100,))  # Binary classification labels
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# --- 2. Define model ---
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 2)  # Output 2 classes

    def forward(self, x):
        return self.fc(x)

model = SimpleNet()

# --- 3. Define loss and optimizer ---
num_iters = 1000

criterion = nn.CrossEntropyLoss()
base_lr = 0.01
warmup_steps = 100
# step-based learning rate scheduler with warmup + cosine annealing
optimizer = optim.Adam(model.parameters(), lr=base_lr)
lr_scheduler1 = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=warmup_steps)
lr_scheduler2 = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_iters - warmup_steps, eta_min=1e-5)
lr_scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[lr_scheduler1, lr_scheduler2], milestones=[warmup_steps])

# --- 4. Training loop ---
iteration = 0
while iteration < num_iters:
    for batch_X, batch_y in dataloader:
        if iteration >= num_iters:
            break

        # 1) Zero gradients from previous step
        optimizer.zero_grad()

        # 2) Forward pass
        outputs = model(batch_X)

        # 3) Compute loss
        loss = criterion(outputs, batch_y)

        # 4) Backward pass
        loss.backward()

        # 5) Update parameters
        optimizer.step()

        # Update learning rate per iteration
        lr_scheduler.step()

        iteration += 1

        # log every 100 iterations
        if iteration % 50 == 0:
            print(f"Iteration [{iteration}/{num_iters}], Loss: {loss.item():.4f}")
            print(f"Current learning rate: {lr_scheduler.get_last_lr()[0]:.6f}")
print("Training complete.")

Iteration [50/1000], Loss: 0.6911
Current learning rate: 0.005500
Iteration [100/1000], Loss: 0.6032
Current learning rate: 0.010000
Iteration [150/1000], Loss: 0.7214
Current learning rate: 0.009924
Iteration [200/1000], Loss: 0.5809
Current learning rate: 0.009699
Iteration [250/1000], Loss: 0.5968
Current learning rate: 0.009331
Iteration [300/1000], Loss: 0.6859
Current learning rate: 0.008831
Iteration [350/1000], Loss: 0.7351
Current learning rate: 0.008216
Iteration [400/1000], Loss: 0.6666
Current learning rate: 0.007503
Iteration [450/1000], Loss: 0.7671
Current learning rate: 0.006713
Iteration [500/1000], Loss: 0.7440
Current learning rate: 0.005872
Iteration [550/1000], Loss: 0.6546
Current learning rate: 0.005005
Iteration [600/1000], Loss: 0.6627
Current learning rate: 0.004138
Iteration [650/1000], Loss: 0.7376
Current learning rate: 0.003297
Iteration [700/1000], Loss: 0.6592
Current learning rate: 0.002508
Iteration [750/1000], Loss: 0.6206
Current learning rate: 0.001

## 2. Training with Gradient Accumulation

In [31]:
# Dummy dataset
X = torch.randn(100, 10)
y = torch.randint(0, 2, (100,))
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# Model
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 2)
    def forward(self, x):
        return self.fc(x)

model = SimpleNet()
criterion = nn.CrossEntropyLoss()
base_lr = 0.01
optimizer = optim.Adam(model.parameters(), lr=base_lr)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-5)

# --- Gradient accumulation parameters ---
accum_steps = 4  # Number of batches to accumulate before update
num_epochs = 5

for epoch in range(num_epochs):
    running_loss = 0.0

    # Zero gradients at the start of the epoch
    optimizer.zero_grad()

    for i, (batch_X, batch_y) in enumerate(dataloader):
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        # Scale loss so total gradient matches large-batch training
        loss = loss / accum_steps
        loss.backward()
        
        # Step optimizer every accum_steps
        if (i + 1) % accum_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        running_loss += loss.item() * accum_steps  # undo scaling for logging
    lr_scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], LR: {lr_scheduler.get_last_lr()[0]:.6f}, Loss: {running_loss/len(dataloader):.4f}")

print("Training complete.")


Epoch [1/5], LR: 0.009046, Loss: 0.7282
Epoch [2/5], LR: 0.006549, Loss: 0.7277
Epoch [3/5], LR: 0.003461, Loss: 0.7068
Epoch [4/5], LR: 0.000964, Loss: 0.7009
Epoch [5/5], LR: 0.000010, Loss: 0.7286
Training complete.


## 3. Mixed Precision Training
- use `torch.cuda.amp` and `GradScaler`

```scss
       ┌───────────────────────────────────────────────────────────────┐
       │                        Forward Pass                            │
       └───────────────────────────────────────────────────────────────┘
               │
               ▼
       Inputs (FP32 from dataset)
               │
               ▼
    ┌───────────────────┐
    │ autocast() ON     │
    │   - MatMul, Conv  → FP16  (fast on Tensor Cores)
    │   - Softmax, BN   → FP32  (for stability)
    └───────────────────┘
               │
               ▼
      Model Outputs (mixed: mostly FP16)
               │
               ▼
        Loss computation (FP32)
               │
               ▼
       ┌────────────────────────────┐
       │ GradScaler: multiply loss  │   ← scale factor (e.g., 1024)
       └────────────────────────────┘
               │
               ▼
       Scaled loss (FP32)
               │
               ▼
       Backward pass (gradients mostly FP16)
               │
               ▼
       ┌───────────────────────────────────────────┐
       │ GradScaler: unscale gradients to FP32     │
       │   - Checks for NaN/Inf → adjust scale     │
       └───────────────────────────────────────────┘
               │
               ▼
       Optimizer step in FP32 (weights stored in FP32)
               │
               ▼
       Updated weights
```

Let's usea transformer model as an example:

| Variable / Tensor Type                   | Stored as             | Why                                                          |
| ---------------------------------------- | --------------------- | ------------------------------------------------------------ |
| **Model master weights**                 | FP32                  | Prevents small updates from being lost; numerical stability. |
| **Model weights in forward pass**        | FP16 (temporary cast) | Allows Tensor Core acceleration, lower memory usage.         |
| **Activations (intermediate outputs)**   | FP16 (mostly)         | Safe for speed/memory, except some sensitive ops.            |
| **Gradients (raw)**                      | FP16                  | Matches activation precision for backprop.                   |
| **Gradient accumulation buffer**         | FP32                  | Accumulates gradients without precision loss.                |
| **Optimizer state** (e.g., Adam moments) | FP32                  | Keeps accuracy for long-term training stability.             |
| **Loss value**                           | FP32                  | Avoids overflow in final scalar.                             |

In a forward pass:
```css

[FP16]  Input embeddings
   │
   ▼
[FP32 inside] LayerNorm → output in FP16
   │
   ▼
[FP16] Q/K/V MatMul (weights cast from FP32 → FP16)
   │
   ▼
[FP32] Softmax over attention scores
   │
   ▼
[FP16] Attention * Value matmul
   │
   ▼
[FP16] Output projection matmul
   │
   ▼
[FP16] Residual add
   │
   ▼
[FP32 inside] LayerNorm → output in FP16
   │
   ▼
[FP16] Linear → GeLU → Linear
   │
   ▼
[FP16] Residual add
   │
   ▼
[FP32 inside] LayerNorm → output in FP16

```


In a backward pass: because the gradients are in FP16, there is a risk of underflow when gradient is small. Therefore usually need scale the gradient up when backpropagating:
- Scale the loss, which scales all the gradients up during backpropagation.
- Backward runs in FP16 for most ops (matching forward precision).
- Any reductions (like sum of gradients) are done in FP32 to avoid loss of precision.
- Gradients for each parameter are stored in FP16 temporarily.
- Before optimizer update:
  - Gradients are unscaled (by GradScaler) and cast to FP32.
  - Optimizer states and master weights are updated in FP32.

In [ ]:
# Dummy dataset
X = torch.randn(100, 10)
y = torch.randint(0, 2, (100,))
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# Model
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 2)
    def forward(self, x):
        return self.fc(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# --- Mixed precision and accumulation ---
scaler = torch.amp.GradScaler("cuda")
accum_steps = 4
num_epochs = 5

for epoch in range(num_epochs):
    running_loss = 0.0
    optimizer.zero_grad()

    for i, (batch_X, batch_y) in enumerate(dataloader):
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        with torch.amp.autocast("cuda"):
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss = loss / accum_steps  # scale for accumulation

        # backward pass with scaled loss
        scaler.scale(loss).backward()

        if (i + 1) % accum_steps == 0:
            scaler.step(optimizer)       # unscale grads + optimizer step
            scaler.update()              # update scaling factor
            optimizer.zero_grad()

        running_loss += loss.item() * accum_steps  # for logging

    # Handle leftover batches (if len(dataloader) % accum_steps != 0)
    if len(dataloader) % accum_steps != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(dataloader):.4f}")

print("Training complete.")

Epoch [1/5], Loss: 0.8886
Epoch [2/5], Loss: 0.8283
Epoch [3/5], Loss: 0.8865
Epoch [4/5], Loss: 0.8651
Epoch [5/5], Loss: 0.9221
Training complete.


/var/folders/_z/2crxc14s6s7g59q3y2sf8gy80000gp/T/ipykernel_78848/2962178639.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/var/folders/_z/2crxc14s6s7g59q3y2sf8gy80000gp/T/ipykernel_78848/2962178639.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Note, if using BF16, which has the same exponent range as F32, there is no underflow risk, thus there is no need to use GradScaler.

## 4. Data Parallelism

The following assumes a multi-GPU setup using PyTorch's DistributedDataParallel (DDP):


In [33]:
# train_ddp.py
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data.distributed import DistributedSampler

def setup(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def cleanup():
    dist.destroy_process_group()

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 2)
    def forward(self, x):
        return self.fc(x)

def train(rank, world_size):
    print(f"Running on rank {rank}.")
    setup(rank, world_size)

    # Dummy dataset
    X = torch.randn(100, 10)
    y = torch.randint(0, 2, (100,))
    dataset = TensorDataset(X, y)
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True) # split data into different subsets for each GPU
    dataloader = DataLoader(dataset, batch_size=16, sampler=sampler)

    # Model: put model in current GPU
    model = SimpleNet().cuda(rank)
    ddp_model = DDP(model, device_ids=[rank])

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(ddp_model.parameters(), lr=0.001)

    # Training loop
    for epoch in range(5):
        sampler.set_epoch(epoch)  # shuffle differently each epoch
        running_loss = 0.0
        for batch_X, batch_y in dataloader:
            batch_X, batch_y = batch_X.cuda(rank), batch_y.cuda(rank)
            optimizer.zero_grad()
            outputs = ddp_model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Rank {rank}, Epoch {epoch}, Loss: {running_loss/len(dataloader):.4f}")

    cleanup()

if __name__ == "__main__":
    import torch.multiprocessing as mp
    world_size = torch.cuda.device_count()
    mp.spawn(train,
             args=(world_size,),
             nprocs=world_size,
             join=True)


### Gradient Synchronization in Distributed Training

**The Problem**: Each GPU computes gradients on different data batches, leading to different gradient values. Without synchronization, models would diverge.

**The Solution**: **All-Reduce** operation that combines gradients from all GPUs.

#### How Gradient Synchronization Works:

```
GPU 0: computes grad_0 on batch_0
GPU 1: computes grad_1 on batch_1  
GPU 2: computes grad_2 on batch_2
GPU 3: computes grad_3 on batch_3

     ↓ All-Reduce (sum + average)

All GPUs: final_grad = (grad_0 + grad_1 + grad_2 + grad_3) / 4
```

#### All-Reduce Communication Pattern:

**Ring All-Reduce** (most common):
```
Step 1: Each GPU sends to next, receives from previous
GPU 0 → GPU 1 → GPU 2 → GPU 3 → GPU 0

Step 2: Reduce (sum) received chunks
Step 3: Scatter-gather to distribute final results
```

**Tree All-Reduce**:
```
    GPU 0
   ╱     ╲
GPU 1    GPU 2
          │
        GPU 3

Hierarchical reduction, then broadcast back down
```

#### When Synchronization Happens:

1. **Forward Pass**: Each GPU processes its own batch independently
2. **Backward Pass**: Each GPU computes gradients independently  
3. **Gradient Sync**: **DDP automatically inserts all-reduce hooks**
4. **Parameter Update**: All GPUs use the synchronized gradients

#### Key Points:

- **Automatic**: DDP handles synchronization automatically
- **Efficient**: Only gradients are communicated, not activations
- **Overlapped**: Gradient sync can overlap with backward computation
- **Bucket-based**: Gradients are grouped into buckets for efficient communication

In [34]:
# Manual Gradient Synchronization (for educational purposes)
# DDP does this automatically, but here's how it works under the hood

import torch
import torch.distributed as dist

def manual_all_reduce_gradients(model, world_size):
    """
    Manually synchronize gradients across all GPUs
    This is what DDP does automatically
    """
    for param in model.parameters():
        if param.grad is not None:
            # Sum gradients from all processes
            dist.all_reduce(param.grad.data, op=dist.ReduceOp.SUM)
            # Average by world size
            param.grad.data /= world_size

# Example of DDP gradient synchronization behavior
def demonstrate_gradient_sync():
    """
    Demonstrate how gradients are synchronized in DDP
    """
    
    # Simulate different gradients on different GPUs
    print("Before synchronization:")
    print("GPU 0 gradient: [1.0, 2.0, 3.0]")
    print("GPU 1 gradient: [2.0, 1.0, 4.0]") 
    print("GPU 2 gradient: [3.0, 3.0, 2.0]")
    print("GPU 3 gradient: [4.0, 4.0, 1.0]")
    
    # Simulate all-reduce operation
    gradients = torch.tensor([
        [1.0, 2.0, 3.0],  # GPU 0
        [2.0, 1.0, 4.0],  # GPU 1
        [3.0, 3.0, 2.0],  # GPU 2
        [4.0, 4.0, 1.0]   # GPU 3
    ])
    
    # All-reduce: sum then average
    synchronized_grad = gradients.mean(dim=0)
    
    print("\\nAfter all-reduce synchronization:")
    print(f"All GPUs gradient: {synchronized_grad.tolist()}")
    print(f"Average: {synchronized_grad}")

# Run demonstration
demonstrate_gradient_sync()

Before synchronization:
GPU 0 gradient: [1.0, 2.0, 3.0]
GPU 1 gradient: [2.0, 1.0, 4.0]
GPU 2 gradient: [3.0, 3.0, 2.0]
GPU 3 gradient: [4.0, 4.0, 1.0]
\nAfter all-reduce synchronization:
All GPUs gradient: [2.5, 2.5, 2.5]
Average: tensor([2.5000, 2.5000, 2.5000])


In [35]:
# Enhanced DDP Training with Gradient Synchronization Details
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data.distributed import DistributedSampler

def setup_distributed(rank, world_size, backend='nccl'):
    """Initialize distributed training"""
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    
    # Initialize process group
    dist.init_process_group(backend, rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)
    
    print(f"Initialized process group: rank {rank}/{world_size}")

def cleanup_distributed():
    """Clean up distributed training"""
    dist.destroy_process_group()

class DetailedDDPTrainer:
    def __init__(self, rank, world_size):
        self.rank = rank
        self.world_size = world_size
        self.device = torch.cuda.device(rank)
        
        # Setup distributed training
        setup_distributed(rank, world_size)
        
        # Create model and wrap with DDP
        self.model = SimpleNet().cuda(rank)
        
        # DDP wrapper - this enables automatic gradient synchronization
        self.ddp_model = DDP(
            self.model, 
            device_ids=[rank],
            find_unused_parameters=False,  # Set to True if some parameters don't receive gradients
            gradient_as_bucket_view=True,  # Memory optimization
            broadcast_buffers=True,        # Synchronize buffers (like BatchNorm running stats)
            bucket_cap_mb=25              # Bucket size for gradient communication
        )
        
        # Print DDP configuration
        if rank == 0:
            print(f"DDP Configuration:")
            print(f"  - Bucket cap: {25} MB")
            print(f"  - Backend: nccl")
            print(f"  - World size: {world_size}")
    
    def create_dataloader(self, batch_size=16):
        """Create distributed dataloader"""
        # Dummy dataset
        X = torch.randn(1000, 10)
        y = torch.randint(0, 2, (1000,))
        dataset = TensorDataset(X, y)
        
        # Distributed sampler ensures each GPU gets different data
        sampler = DistributedSampler(
            dataset, 
            num_replicas=self.world_size, 
            rank=self.rank, 
            shuffle=True
        )
        
        dataloader = DataLoader(
            dataset, 
            batch_size=batch_size, 
            sampler=sampler,
            pin_memory=True,  # Faster GPU transfer
            num_workers=2
        )
        
        return dataloader, sampler
    
    def train_step(self, batch_X, batch_y, optimizer, criterion):
        """Single training step with gradient sync details"""
        
        # 1. Forward pass (independent on each GPU)
        outputs = self.ddp_model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # 2. Backward pass (independent gradient computation)
        loss.backward()
        
        # 3. DDP automatically synchronizes gradients here!
        # This happens when loss.backward() completes
        # DDP uses hooks registered on parameters to trigger all-reduce
        
        return loss.item()
    
    def train(self, num_epochs=3, lr=0.001):
        """Full training loop with detailed logging"""
        
        dataloader, sampler = self.create_dataloader()
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.ddp_model.parameters(), lr=lr)
        
        for epoch in range(num_epochs):
            # Important: set epoch for proper shuffling
            sampler.set_epoch(epoch)
            
            running_loss = 0.0
            num_batches = 0
            
            for batch_idx, (batch_X, batch_y) in enumerate(dataloader):
                batch_X = batch_X.cuda(self.rank, non_blocking=True)
                batch_y = batch_y.cuda(self.rank, non_blocking=True)
                
                optimizer.zero_grad()
                
                # Training step with automatic gradient sync
                loss = self.train_step(batch_X, batch_y, optimizer, criterion)
                
                # Optimizer step uses synchronized gradients
                optimizer.step()
                
                running_loss += loss
                num_batches += 1
                
                # Log gradient norms (should be identical across GPUs)
                if batch_idx == 0 and self.rank == 0:
                    total_norm = 0
                    for p in self.ddp_model.parameters():
                        if p.grad is not None:
                            param_norm = p.grad.data.norm(2)
                            total_norm += param_norm.item() ** 2
                    total_norm = total_norm ** (1. / 2)
                    print(f"Epoch {epoch}, Gradient norm: {total_norm:.4f}")
            
            avg_loss = running_loss / num_batches
            
            # Synchronize loss across GPUs for consistent logging
            loss_tensor = torch.tensor(avg_loss).cuda(self.rank)
            dist.all_reduce(loss_tensor, op=dist.ReduceOp.SUM)
            synchronized_loss = loss_tensor.item() / self.world_size
            
            if self.rank == 0:
                print(f"Epoch {epoch+1}, Synchronized Loss: {synchronized_loss:.4f}")
        
        cleanup_distributed()

# Function to run distributed training
def run_distributed_training(rank, world_size):
    try:
        trainer = DetailedDDPTrainer(rank, world_size)
        trainer.train()
    except Exception as e:
        print(f"Error in rank {rank}: {e}")
        cleanup_distributed()

# To run this (in practice, you'd use torchrun or mp.spawn):
# if __name__ == "__main__":
#     import torch.multiprocessing as mp
#     world_size = torch.cuda.device_count()
#     if world_size > 1:
#         mp.spawn(run_distributed_training, args=(world_size,), nprocs=world_size, join=True)
#     else:
#         print("Need multiple GPUs for distributed training")

### Advanced Gradient Synchronization Strategies

#### 1. **Gradient Compression**
Reduce communication overhead by compressing gradients:

```python
# Example: Quantization-based compression
def compress_gradients(gradients, bits=8):
    # Quantize gradients to reduce communication
    min_val, max_val = gradients.min(), gradients.max()
    scale = (2**bits - 1) / (max_val - min_val)
    quantized = torch.round((gradients - min_val) * scale)
    return quantized, min_val, scale

def decompress_gradients(quantized, min_val, scale):
    return quantized / scale + min_val
```

#### 2. **Gradient Accumulation with Sync Control**
Control when synchronization happens:

```python
class SyncControlDDP(nn.Module):
    def __init__(self, model, sync_every_n_steps=4):
        super().__init__()
        self.ddp_model = DDP(model)
        self.sync_every_n_steps = sync_every_n_steps
        self.step_count = 0
    
    def forward(self, x):
        # Disable automatic sync
        if self.step_count % self.sync_every_n_steps != 0:
            with self.ddp_model.no_sync():
                return self.ddp_model(x)
        else:
            # Normal forward with sync
            return self.ddp_model(x)
```

#### 3. **Hierarchical All-Reduce**
For large clusters, use hierarchical communication:

```
Node 0          Node 1          Node 2
GPU0 GPU1       GPU2 GPU3       GPU4 GPU5
 ↓    ↓          ↓    ↓          ↓    ↓
Intra-node All-Reduce (fast NVLink/IB)
     ↓              ↓               ↓
Inter-node All-Reduce (slower network)
     ↓              ↓               ↓
Broadcast back within nodes
```

#### 4. **Asynchronous Parameter Updates**
Alternative to synchronous all-reduce:

- **Parameter Servers**: Central servers hold parameters
- **Gossip Algorithms**: Peer-to-peer parameter sharing
- **Local SGD**: Update locally, sync periodically

#### 5. **Communication Backend Comparison**

| Backend | Best For | Pros | Cons |
|---------|----------|------|------|
| **NCCL** | NVIDIA GPUs | Optimized for GPU-GPU | NVIDIA only |
| **Gloo** | CPUs, Mixed | Cross-platform | Slower for GPUs |
| **MPI** | HPC clusters | Mature, flexible | Complex setup |

#### 6. **Memory and Bandwidth Optimization**

```python
# DDP Optimizations
ddp_model = DDP(
    model,
    gradient_as_bucket_view=True,    # Reduce memory copies
    broadcast_buffers=True,          # Sync batch norm stats
    find_unused_parameters=False,    # Skip unused params (faster)
    bucket_cap_mb=25,               # Larger buckets = fewer allreduce calls
    static_graph=True               # If model structure is fixed
)
```

In [36]:
# Debugging and Monitoring Gradient Synchronization

import torch
import torch.distributed as dist
import time
from collections import defaultdict

class GradientSyncMonitor:
    """Monitor gradient synchronization in distributed training"""
    
    def __init__(self, model, rank, world_size):
        self.model = model
        self.rank = rank
        self.world_size = world_size
        self.sync_times = []
        self.gradient_stats = defaultdict(list)
        
    def check_gradient_consistency(self):
        """Verify gradients are identical across all GPUs"""
        if self.world_size <= 1:
            return True
            
        all_consistent = True
        for name, param in self.model.named_parameters():
            if param.grad is not None:
                # Create tensor for all-gather
                grad_list = [torch.zeros_like(param.grad) for _ in range(self.world_size)]
                
                # Gather gradients from all processes
                dist.all_gather(grad_list, param.grad)
                
                # Check if all gradients are identical
                for i in range(1, self.world_size):
                    if not torch.allclose(grad_list[0], grad_list[i], rtol=1e-5):
                        if self.rank == 0:
                            print(f"WARNING: Gradient mismatch in {name}")
                            print(f"  Rank 0: {grad_list[0].norm().item():.6f}")
                            print(f"  Rank {i}: {grad_list[i].norm().item():.6f}")
                        all_consistent = False
        
        return all_consistent
    
    def measure_sync_time(self, sync_fn):
        """Measure time taken for gradient synchronization"""
        start_time = time.time()
        sync_fn()
        torch.cuda.synchronize()  # Wait for GPU operations to complete
        sync_time = time.time() - start_time
        
        self.sync_times.append(sync_time)
        return sync_time
    
    def compute_gradient_stats(self):
        """Compute statistics about gradients"""
        total_params = 0
        total_grad_norm = 0
        layer_stats = {}
        
        for name, param in self.model.named_parameters():
            if param.grad is not None:
                grad_norm = param.grad.norm().item()
                param_count = param.numel()
                
                total_params += param_count
                total_grad_norm += grad_norm ** 2
                
                layer_stats[name] = {
                    'grad_norm': grad_norm,
                    'param_count': param_count,
                    'grad_mean': param.grad.mean().item(),
                    'grad_std': param.grad.std().item()
                }
        
        total_grad_norm = total_grad_norm ** 0.5
        
        return {
            'total_grad_norm': total_grad_norm,
            'total_params': total_params,
            'layer_stats': layer_stats,
            'avg_sync_time': sum(self.sync_times) / len(self.sync_times) if self.sync_times else 0
        }

# Example: Monitoring gradient sync in training loop
def monitored_training_step(model, batch_X, batch_y, optimizer, criterion, monitor):
    """Training step with gradient synchronization monitoring"""
    
    # Forward pass
    outputs = model(batch_X)
    loss = criterion(outputs, batch_y)
    
    # Backward pass
    loss.backward()
    
    # Check gradient consistency (expensive, use sparingly)
    if hasattr(monitor, '_check_every') and monitor._check_every > 0:
        if not hasattr(monitor, '_step_count'):
            monitor._step_count = 0
        monitor._step_count += 1
        
        if monitor._step_count % monitor._check_every == 0:
            consistent = monitor.check_gradient_consistency()
            if monitor.rank == 0:
                print(f"Step {monitor._step_count}: Gradients consistent = {consistent}")
    
    # Measure synchronization time (DDP handles this automatically)
    def dummy_sync():
        pass  # DDP already synced during backward()
    
    sync_time = monitor.measure_sync_time(dummy_sync)
    
    # Compute gradient statistics
    stats = monitor.compute_gradient_stats()
    
    if monitor.rank == 0 and hasattr(monitor, '_log_every'):
        if monitor._step_count % getattr(monitor, '_log_every', 10) == 0:
            print(f"Step {monitor._step_count}:")
            print(f"  Gradient norm: {stats['total_grad_norm']:.6f}")
            print(f"  Sync time: {sync_time*1000:.2f}ms")
    
    # Optimizer step
    optimizer.step()
    
    return loss.item(), stats

# Communication profiling
class CommProfiler:
    """Profile communication patterns in distributed training"""
    
    def __init__(self, rank, world_size):
        self.rank = rank
        self.world_size = world_size
        self.comm_log = []
    
    def profile_allreduce(self, tensor_size_mb):
        """Profile all-reduce operation"""
        # Create dummy tensor
        elements = int(tensor_size_mb * 1024 * 1024 / 4)  # Assume float32
        dummy_tensor = torch.randn(elements).cuda()
        
        # Warm up
        for _ in range(3):
            dist.all_reduce(dummy_tensor)
            torch.cuda.synchronize()
        
        # Actual measurement
        start_time = time.time()
        dist.all_reduce(dummy_tensor)
        torch.cuda.synchronize()
        end_time = time.time()
        
        bandwidth_mbps = tensor_size_mb / (end_time - start_time)
        
        self.comm_log.append({
            'operation': 'all_reduce',
            'size_mb': tensor_size_mb,
            'time_ms': (end_time - start_time) * 1000,
            'bandwidth_mbps': bandwidth_mbps
        })
        
        return bandwidth_mbps
    
    def get_communication_summary(self):
        """Get summary of communication performance"""
        if not self.comm_log:
            return "No communication data recorded"
        
        total_time = sum(entry['time_ms'] for entry in self.comm_log)
        total_size = sum(entry['size_mb'] for entry in self.comm_log)
        avg_bandwidth = sum(entry['bandwidth_mbps'] for entry in self.comm_log) / len(self.comm_log)
        
        return {
            'total_comm_time_ms': total_time,
            'total_data_mb': total_size,
            'avg_bandwidth_mbps': avg_bandwidth,
            'num_operations': len(self.comm_log)
        }

# Example usage:
def demonstrate_sync_monitoring():
    """Demonstrate gradient sync monitoring"""
    print("Gradient Synchronization Demo")
    print("=" * 40)
    
    # Simulate gradient values on different GPUs
    torch.manual_seed(42)
    
    # Before sync (different gradients)
    grad_gpu0 = torch.randn(1000) * 0.01
    grad_gpu1 = torch.randn(1000) * 0.01
    grad_gpu2 = torch.randn(1000) * 0.01
    
    print("Before synchronization:")
    print(f"GPU 0 gradient norm: {grad_gpu0.norm().item():.6f}")
    print(f"GPU 1 gradient norm: {grad_gpu1.norm().item():.6f}")
    print(f"GPU 2 gradient norm: {grad_gpu2.norm().item():.6f}")
    
    # Simulate all-reduce
    all_grads = torch.stack([grad_gpu0, grad_gpu1, grad_gpu2])
    synced_grad = all_grads.mean(dim=0)
    
    print("\\nAfter synchronization (all-reduce):")
    print(f"All GPUs gradient norm: {synced_grad.norm().item():.6f}")
    
    # Communication overhead estimation
    param_size_mb = synced_grad.numel() * 4 / (1024 * 1024)  # float32
    print(f"\\nCommunication overhead:")
    print(f"Parameter size: {param_size_mb:.2f} MB")
    print(f"All-reduce data: {param_size_mb * 2:.2f} MB")  # Rough estimate
    print("(Actual overhead depends on algorithm and network)")

# Run demonstration
demonstrate_sync_monitoring()

Gradient Synchronization Demo
Before synchronization:
GPU 0 gradient norm: 0.317166
GPU 1 gradient norm: 0.326275
GPU 2 gradient norm: 0.315270
\nAfter synchronization (all-reduce):
All GPUs gradient norm: 0.186314
\nCommunication overhead:
Parameter size: 0.00 MB
All-reduce data: 0.01 MB
(Actual overhead depends on algorithm and network)


## Summary: Gradient Synchronization in Multi-GPU Training

### 🎯 **Key Takeaways**

| Aspect | Details |
|--------|---------|
| **Purpose** | Ensure all GPUs have identical gradients for consistent parameter updates |
| **Mechanism** | **All-Reduce** operation (sum + average across all GPUs) |
| **When** | Automatically after `loss.backward()` in DDP |
| **Communication** | Only gradients are synchronized, not activations |
| **Algorithms** | Ring All-Reduce (most common), Tree All-Reduce, Hierarchical |

### ⚡ **How DDP Handles It Automatically**

```python
# What happens inside DDP during training:

# 1. Forward pass - each GPU processes different data
outputs = model(batch_X)  # Independent on each GPU

# 2. Compute loss
loss = criterion(outputs, batch_y)  # Independent on each GPU

# 3. Backward pass + Auto Gradient Sync
loss.backward()  # 🔥 DDP automatically triggers all-reduce here!

# 4. Parameter update with synchronized gradients
optimizer.step()  # All GPUs use identical gradients
```

### 🔧 **Performance Optimizations**

1. **Overlapped Communication**: Gradient sync overlaps with backward computation
2. **Gradient Buckets**: Groups parameters for efficient batch communication  
3. **Memory Views**: Reduces memory copies with `gradient_as_bucket_view=True`
4. **Compression**: Advanced techniques can compress gradients before sync

### 🚨 **Common Issues & Solutions**

| Problem | Cause | Solution |
|---------|--------|----------|
| **Out of sync models** | Manual parameter updates | Use DDP wrapper only |
| **Slow training** | Large parameter sync | Optimize bucket size, use gradient compression |
| **Memory errors** | Multiple model copies | Use `gradient_as_bucket_view=True` |
| **Unused parameters** | Dynamic models | Set `find_unused_parameters=True` |

### 📊 **Communication Cost**

For a model with P parameters:
- **Data transferred per step**: ~2P (forward + backward gradients)  
- **Time complexity**: O(P) for Ring All-Reduce
- **Bandwidth required**: Depends on model size and update frequency

**Example**: For a 1B parameter model (4GB), each sync requires ~8GB of communication across the cluster.

### 🎛️ **Advanced Control**

```python
# Manual control over synchronization
with ddp_model.no_sync():
    # Skip synchronization for this step
    loss.backward()

# Synchronize manually when needed
ddp_model._sync_params_and_buffers()
```

The beauty of DDP is that gradient synchronization happens **automatically and efficiently**, ensuring all GPUs stay perfectly in sync without any manual intervention!